In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import joblib

preprocessor = joblib.load('preprocessor.pkl')
X_train_processed = joblib.load('X_train_processed.pkl')
X_test_processed = joblib.load('X_test_processed.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')
scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline = Pipeline(steps=[
    ('model', XGBClassifier(n_estimators=100, random_state=42))
])


param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05],
    'model__max_depth': [3],
    'model__subsample': [0.8],
    'model__min_child_weight': [2, 5, 10],
    'model__reg_alpha': [0, 0.1, 0.5],
    'model__reg_lambda': [1, 2, 5, 10]
}

grid_search = GridSearchCV(pipeline, param_grid=param_grid, scoring='average_precision', cv=5, n_jobs=-1)

grid_search.fit(X_train_processed, y_train)

print('Best Params', grid_search.best_params_)
print('Best PR-AUC', grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
y_pred = best_model.predict(X_test_processed)

pr_auc = average_precision_score(y_test, y_pred_proba)
print(classification_report(y_test, y_pred))

train_proba = best_model.predict_proba(X_train_processed)[:, 1]
y_pred_train = best_model.predict(X_train_processed)

print('Train PR-AUC:', average_precision_score(y_train, train_proba))
print('Test PR-AUC:', average_precision_score(y_test, y_pred_proba))
print('\nTrain Classification Report:')
print(classification_report(y_train, y_pred_train))
print('\nTest Classification Report:')
print(classification_report(y_test, y_pred))



Best Params {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__min_child_weight': 5, 'model__n_estimators': 100, 'model__reg_alpha': 0.5, 'model__reg_lambda': 1, 'model__subsample': 0.8}
Best PR-AUC 0.7138834794716873
              precision    recall  f1-score   support

           0       0.78      0.95      0.86       151
           1       0.53      0.16      0.25        49

    accuracy                           0.76       200
   macro avg       0.66      0.56      0.55       200
weighted avg       0.72      0.76      0.71       200

Train PR-AUC: 0.8259221949906428
Test PR-AUC: 0.6081754660333615

Train Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.99      0.91       602
           1       0.93      0.40      0.56       198

    accuracy                           0.84       800
   macro avg       0.88      0.70      0.73       800
weighted avg       0.86      0.84      0.82       800


Test Classification R

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib

preprocessor = joblib.load('preprocessor.pkl')
X_train_processed = joblib.load('X_train_processed.pkl')
X_test_processed = joblib.load('X_test_processed.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')
scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline = Pipeline(steps=[
    ('model', XGBClassifier(n_estimators=100, scale_pos_weight=scale, random_state=42))
])

pipeline.fit(X_train_processed, y_train)

y_pred_proba = pipeline.predict_proba(X_test_processed)[:, 1]

pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"PR-AUC: {pr_auc:.4f}")
y_pred = pipeline.predict(X_test_processed)
print(classification_report(y_test, y_pred))
print(f"scale_pos_weight: {scale:.2f}")


PR-AUC: 0.5976
              precision    recall  f1-score   support

           0       0.87      0.87      0.87       151
           1       0.60      0.61      0.61        49

    accuracy                           0.81       200
   macro avg       0.74      0.74      0.74       200
weighted avg       0.81      0.81      0.81       200

scale_pos_weight: 3.04
